# Get hydro data

We'll use this data in our streamflow prediction model for East Canyon, UT.

Use the 'hyriver' environment.

In [8]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supportingscripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pydaymet as daymet
import warnings
warnings.filterwarnings("ignore")

## Sites for evaluation

Different sites will be used for LSTM model training, validating, and testing.

In [166]:
# Site #1
station_id = '10164500'
basinname = 'AmericanFork'

In [97]:
# Site #2
station_id = '10133800'
basinname = 'EastCanyonCreek_JeremyRanch'

In [130]:
# Site #3
station_id = '10133600'
basinname = 'McLeodCreek_ParkCity'

In [49]:
# Site #4
station_id = '10128500'
basinname = 'WeberRiver_Oakley'

In [73]:
# Site #5
station_id = '10149400'
basinname = 'DiamondFork_Thistle'

## Download/load all data

#### SNOTEL
 Download SNOTEL data from stations within the basin of USGS station of interest. 

In [167]:
nldi = NLDI()

In [168]:
#Getting basin geometry
print('Collecting basins...', end='')
basin = nldi.get_basins(station_id)
if not os.path.exists('files'):
    os.makedirs('files')
basin.to_file(f"files/{basinname}.shp")
print('done')

site_feature = nldi.getfeature_byid("nwissite", f"USGS-{station_id}")
upstream_network = nldi.navigate_byid(
    "nwissite", f"USGS-{station_id}", "upstreamMain", "flowlines", distance=9999
)

In [169]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = all_stations_gdf[all_stations_gdf.geometry.within(basin.geometry.iloc[0])]

#reset index to have siteid as a column
gdf_in_bbox.reset_index(drop=False, inplace=True)

#make begin and end date a str
gdf_in_bbox['beginDate'] = [datetime.datetime.strftime(gdf_in_bbox['beginDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox['endDate'] = [datetime.datetime.strftime(gdf_in_bbox['endDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox

,code,name,network,elevation_m,latitude,longitude,state,HUC,mgrs,mountainRange,beginDate,endDate,csvData,geometry
0,820_UT_SNTL,Timpanogos Divide,SNOTEL,2481.072021,40.428169,-111.616333,Utah,160202010802,12TVK,Western Rocky Mountains,1978-10-01,2026-04-14,True,POINT (-111.61633 40.42817)


In [170]:
# Use the getData module to retrieve data 
OutputFolder = f'files/SNOTEL/{station_id}/'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)

for i in gdf_in_bbox.index:
    getData.getSNOTELData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], gdf_in_bbox.state[i], gdf_in_bbox.beginDate[i], gdf_in_bbox.endDate[i], OutputFolder)


Start retrieving data for Timpanogos Divide, 820_UT_SNTL 
 https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customMultiTimeSeriesGroupByStationReport/daily/start_of_period/820:Ut:SNTL%7Cid=%22%22%7Cname/1978-10-01,2026-04-14/WTEQ::value?fitToScreen=false


In [171]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = f'files/SNOTEL/{station_id}/'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
   

820: 17363 start date: 1978-10-01 00:00:00 end date: 2026-04-14 00:00:00


In [172]:
#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.head()

,820_SWE_cm
Date,
1978-10-01,0.0
1978-10-02,0.0
1978-10-03,0.0
1978-10-04,0.0
1978-10-05,0.0


#### Daymet

Download met data from Daymet

In [173]:
nldi = NLDI()

# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
start_date = "1980-10-01" # Start of water year, started with a smaller range
end_date = datetime.datetime.today().strftime('%Y-%m-%d') # End date is today, but could be set to the end of the water year
var = ["prcp", "tmin", "tmax",'srad', 'swe', 'vp', 'dayl'] # Variables to fetch, precip, temperature, solar radiation, snow water equivalent, vapor pressure, day length
dates = (start_date, end_date ) # Started with a smaller range to test

In [174]:
#Get geometry and ensure CRS is correct
basin = NLDI().get_basins(station_id)
geometry_centroid = geometry.centroid
centroid = (geometry_centroid.x, geometry_centroid.y)


#Fetch data - authentication now happens automatically via earthaccess/.netrc
# Try this simplified call first
met_df = daymet.get_bycoords(centroid, dates, variables=var)

met_df.head()

,dayl (s),prcp (mm/day),srad (W/m2),swe (kg/m2),tmax (degrees C),tmin (degrees C),vp (Pa)
time,,,,,,,
1980-01-01,33000.78,4.50,92.79,39.85,1.03,-3.26,480.17
1980-01-02,33042.81,4.22,152.35,44.07,0.39,-7.08,359.27
1980-01-03,33088.28,0.00,275.87,44.07,0.64,-10.89,266.47
1980-01-04,33137.14,0.00,274.48,44.07,5.02,-6.55,374.14
1980-01-05,33189.38,0.00,282.73,43.54,7.72,-4.74,429.53


In [175]:
# clean the dataframe, rename the columns
met_df.rename(columns={"prcp (mm/day)": "prcp_mm_day",'srad (W/m2)': "srad_W_m2", 'swe (kg/m2)': "swe_cm", "vp (Pa)": "vp_Pa", "dayl (s)": "dayl_s", "tmin (degrees C)": "tmin_C", "tmax (degrees C)": "tmax_C"}, inplace=True)
#Calculate Mean Temperature
met_df["tmean"] = (met_df.tmax_C + met_df.tmin_C) / 2
#convert swe from kg/m2 to cm, 1 kg/m2 is equivalent to 0.1 cm of water
met_df["swe_cm"] = met_df["swe_cm"] * 0.1

#set the index to name to date
met_df.index.name = "Date"

met_df.head()

,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,tmean
Date,,,,,,,,
1980-01-01,33000.78,4.50,92.79,3.985,1.03,-3.26,480.17,-1.115
1980-01-02,33042.81,4.22,152.35,4.407,0.39,-7.08,359.27,-3.345
1980-01-03,33088.28,0.00,275.87,4.407,0.64,-10.89,266.47,-5.125
1980-01-04,33137.14,0.00,274.48,4.407,5.02,-6.55,374.14,-0.765
1980-01-05,33189.38,0.00,282.73,4.354,7.72,-4.74,429.53,1.490


#### NLDS

Download NLDS data

In [176]:
# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
basin_polygon_coords = list(geometry.exterior.coords)

In [177]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df1 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1980-01-01', 
                                          end_date='1990-01-1')
Daily_NLDAS_df1.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1980-01-01,0.013244,278.689838,5.994014,0.008046,77096.498145,59.288830,0.003436,-2.607362,0.266971,0.787105,-0.097493
1980-01-02,0.000000,264.265524,27.747052,0.010779,77208.987125,80.444898,0.003367,-3.740573,0.063661,1.168021,-0.238665
1980-01-03,0.000000,221.823376,0.000000,0.028093,77163.612135,93.497785,0.002706,-4.899115,0.000000,-0.401634,1.407928
1980-01-04,0.000000,216.878071,0.000000,0.041122,76977.310936,108.581505,0.002228,-3.537013,0.000000,-0.572513,1.812769
1980-01-05,0.000000,265.423193,0.000000,0.051765,76906.156542,85.374115,0.002347,-1.623043,0.007260,-0.596229,2.484022


In [178]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df2 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1990-01-01', 
                                          end_date='2000-01-1')
Daily_NLDAS_df2.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1990-01-01,0.000000,196.766799,0.000000,0.038482,76750.808438,102.200627,0.002020,-5.575502,0.000000,-1.482182,2.425517
1990-01-02,0.141505,253.157148,37.913676,0.038740,76034.258202,91.721945,0.002890,-3.372441,0.389912,0.927722,1.473650
1990-01-03,0.027052,203.617021,3.430088,0.065688,76552.927407,108.933608,0.001912,-8.279139,0.034184,2.694212,-2.852607
1990-01-04,0.000000,174.130941,0.000000,0.031881,77195.405001,106.755734,0.001555,-11.483570,0.278703,0.472959,1.067898
1990-01-05,0.000000,248.311294,0.000000,0.027286,77252.039192,95.010850,0.002326,-7.980791,0.001039,0.797147,0.526039


In [179]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df3 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2000-01-01', 
                                          end_date='2010-01-1')
Daily_NLDAS_df3.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2000-01-01,0.000000,217.281643,0.000000,0.017626,76252.447683,94.348657,0.002264,-7.259574,0.004749,0.567824,0.088924
2000-01-02,0.029455,226.962365,9.335847,0.033167,76168.314524,74.662107,0.002157,-8.374227,0.308612,1.970864,-0.597858
2000-01-03,0.000000,235.709385,4.230299,0.035794,76974.605390,74.767493,0.001909,-10.072329,0.222535,2.719975,-0.367038
2000-01-04,0.000000,205.420782,0.000000,0.024217,77469.615280,59.163013,0.001597,-11.400474,0.000000,-0.904080,2.182594
2000-01-05,0.000000,253.154402,3.956508,0.028578,77049.549735,72.544221,0.002464,-7.415292,0.405585,2.697895,-0.310988


In [180]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df4 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2010-01-01', 
                                          end_date='2020-01-1')
Daily_NLDAS_df4.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2010-01-01,0.000000,257.913956,0.831307,0.024294,77363.065794,75.131144,0.002478,-7.172238,0.079223,-1.259891,2.707578
2010-01-02,0.040828,268.266381,9.072901,0.008499,77261.703855,81.256302,0.003267,-4.641547,0.081102,1.223369,1.261567
2010-01-03,0.000000,194.393029,0.000000,0.002458,77335.706524,105.705348,0.002170,-9.062351,0.004976,0.094318,1.049716
2010-01-04,0.000000,190.624526,0.000000,0.006929,77382.428423,106.260505,0.002023,-9.895573,0.000000,-1.346412,1.354270
2010-01-05,0.000000,228.912376,0.000000,0.008496,77116.780559,71.702890,0.002277,-8.319501,0.000000,-0.476578,1.356637


In [181]:
#combine the four dataframes into one
Daily_NLDAS_df = pd.concat([Daily_NLDAS_df1, Daily_NLDAS_df2, Daily_NLDAS_df3, Daily_NLDAS_df4], ignore_index=False)

Daily_NLDAS_df.head()

,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1980-01-01,0.013244,278.689838,5.994014,0.008046,77096.498145,59.288830,0.003436,-2.607362,0.266971,0.787105,-0.097493
1980-01-02,0.000000,264.265524,27.747052,0.010779,77208.987125,80.444898,0.003367,-3.740573,0.063661,1.168021,-0.238665
1980-01-03,0.000000,221.823376,0.000000,0.028093,77163.612135,93.497785,0.002706,-4.899115,0.000000,-0.401634,1.407928
1980-01-04,0.000000,216.878071,0.000000,0.041122,76977.310936,108.581505,0.002228,-3.537013,0.000000,-0.572513,1.812769
1980-01-05,0.000000,265.423193,0.000000,0.051765,76906.156542,85.374115,0.002347,-1.623043,0.007260,-0.596229,2.484022


#### USGS streamflow

Load streamflow data

In [182]:
#get streamflow information using the NWIS id for the gage and ghte get_usgs_streamflow function in getData.py 
streamflow = getData.get_usgs_streamflow(station_id)

Retrieving data for Site: 10164500 from 1980-01-01 to 2026-04-15...


In [183]:
streamflow.head()

,site_no,00060_Mean,00060_Mean_cd
datetime,,,
1980-01-01 00:00:00+00:00,10164500,16.0,A
1980-01-02 00:00:00+00:00,10164500,16.0,A
1980-01-03 00:00:00+00:00,10164500,16.0,A
1980-01-04 00:00:00+00:00,10164500,16.0,A
1980-01-05 00:00:00+00:00,10164500,16.0,A


In [184]:
streamflow_df = dataprocessing.clean_nwis_dataframe(streamflow)
#set the index name to Date
streamflow_df.index.name = "Date"
streamflow_df.head()

,site_no,flow_cfs
Date,,
1980-01-01,10164500,16.0
1980-01-02,10164500,16.0
1980-01-03,10164500,16.0
1980-01-04,10164500,16.0
1980-01-05,10164500,16.0


#### Now put all the data together

In [185]:
#find the latest start date and the earliest end date for SNOTEL_df, met_df, cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 
end_date = min([df.index.max() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
met_df = met_df[(met_df.index >= begin_date) & (met_df.index <= end_date)]
streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]
Daily_NLDAS_df = Daily_NLDAS_df[(Daily_NLDAS_df.index >= begin_date) & (Daily_NLDAS_df.index <= end_date)]

In [186]:
#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, met_df, Daily_NLDAS_df,streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

,site_no,flow_cfs,820_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
1980-01-01,10164500,16.0,15.494,33000.78,4.50,92.79,3.985,1.03,-3.26,480.17,...,278.689838,5.994014,0.008046,77096.498145,59.288830,0.003436,-2.607362,0.266971,0.787105,-0.097493
1980-01-02,10164500,16.0,15.494,33042.81,4.22,152.35,4.407,0.39,-7.08,359.27,...,264.265524,27.747052,0.010779,77208.987125,80.444898,0.003367,-3.740573,0.063661,1.168021,-0.238665
1980-01-03,10164500,16.0,15.494,33088.28,0.00,275.87,4.407,0.64,-10.89,266.47,...,221.823376,0.000000,0.028093,77163.612135,93.497785,0.002706,-4.899115,0.000000,-0.401634,1.407928
1980-01-04,10164500,16.0,15.494,33137.14,0.00,274.48,4.407,5.02,-6.55,374.14,...,216.878071,0.000000,0.041122,76977.310936,108.581505,0.002228,-3.537013,0.000000,-0.572513,1.812769
1980-01-05,10164500,16.0,15.494,33189.38,0.00,282.73,4.354,7.72,-4.74,429.53,...,265.423193,0.000000,0.051765,76906.156542,85.374115,0.002347,-1.623043,0.007260,-0.596229,2.484022


In [187]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

,site_no,flow_cfs,820_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
1980-01-01,10164500,16.0,15.494,33000.78,4.50,92.79,3.985,1.03,-3.26,480.17,...,278.689838,5.994014,0.008046,77096.498145,59.288830,0.003436,-2.607362,0.266971,0.787105,-0.097493
1980-01-02,10164500,16.0,15.494,33042.81,4.22,152.35,4.407,0.39,-7.08,359.27,...,264.265524,27.747052,0.010779,77208.987125,80.444898,0.003367,-3.740573,0.063661,1.168021,-0.238665
1980-01-03,10164500,16.0,15.494,33088.28,0.00,275.87,4.407,0.64,-10.89,266.47,...,221.823376,0.000000,0.028093,77163.612135,93.497785,0.002706,-4.899115,0.000000,-0.401634,1.407928
1980-01-04,10164500,16.0,15.494,33137.14,0.00,274.48,4.407,5.02,-6.55,374.14,...,216.878071,0.000000,0.041122,76977.310936,108.581505,0.002228,-3.537013,0.000000,-0.572513,1.812769
1980-01-05,10164500,16.0,15.494,33189.38,0.00,282.73,4.354,7.72,-4.74,429.53,...,265.423193,0.000000,0.051765,76906.156542,85.374115,0.002347,-1.623043,0.007260,-0.596229,2.484022


In [188]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

,site_no,flow_cfs,820_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
2019-03-01,10164500,8.46,66.294,39770.06,2.70,232.08,11.467,3.36,-3.19,482.84,...,254.913916,11.009588,0.013598,76749.156708,173.247158,0.003412,-1.534283,0.033116,0.815177,0.343974
2019-03-02,10164500,8.46,66.548,39929.54,12.09,185.93,11.393,0.65,-4.24,446.12,...,273.977041,0.000000,0.032455,76446.954856,126.791326,0.003281,-2.042320,0.379278,-0.424457,-0.198679
2019-03-03,10164500,8.46,67.818,40089.59,9.98,198.51,11.349,0.11,-4.95,422.92,...,275.790892,1.381545,0.016905,76565.383454,124.690142,0.003417,-3.381256,0.385617,0.818999,-1.557878
2019-03-04,10164500,8.46,68.072,40250.20,0.00,371.69,11.349,0.84,-6.53,374.70,...,195.839943,0.449135,0.025870,76805.508663,203.605646,0.002659,-6.421681,0.004690,0.728158,-0.720209
2019-03-05,10164500,8.46,68.072,40411.31,0.00,478.59,11.342,5.27,-5.85,394.88,...,234.036612,0.161037,0.020562,76985.536402,185.341876,0.002799,-4.292509,0.004183,-2.566257,1.618388
2019-03-06,10164500,8.70,69.342,40572.91,24.15,228.46,11.146,4.63,-1.34,553.88,...,275.064011,0.770498,0.002571,76338.403749,130.588002,0.003780,-1.289271,0.429400,-2.155124,2.960984
2019-03-07,10164500,8.92,73.914,40734.97,13.70,205.52,10.947,3.99,-1.25,557.17,...,262.290786,39.343647,-0.002277,76321.787998,162.130216,0.003855,-1.017560,0.383832,0.107459,2.653444
2019-03-08,10164500,8.74,75.946,40897.44,27.90,242.78,10.882,1.78,-4.46,438.80,...,265.028866,27.454992,0.014829,76007.557618,166.306277,0.003536,-1.320789,0.688296,-0.331016,1.148941
2019-03-09,10164500,8.46,78.486,41060.30,3.82,286.89,11.264,-0.02,-7.51,347.47,...,250.296653,22.633239,0.037636,76146.598651,174.632513,0.002878,-4.687610,0.191327,1.307465,0.231194


#### Save the final master data file

Save our data file!

In [189]:
#save data with station ID in the filename
#if HydroDF directory doesn't exist, create it
if not os.path.exists("files/HydroDF"):
    os.makedirs("files/HydroDF")
Hydro_df.to_csv(f"files/HydroDF/HydroDF_{basinname}_{station_id}.csv")

## Create master DF for LSTM

Once all the data has been acquired, merge dfs into a single master for the LSTM

In [ ]:
#in each of the HydroDF csvs, rename the 4th column to SWE_cm
for filename in os.listdir("files/HydroDF/"):
        placeholder = pd.read_csv(os.path.join("files/HydroDF/", filename))
        print(placeholder.columns)
        placeholder.rename(columns={placeholder.columns[3]: "SWE_cm"}, inplace=True)
        print(placeholder.columns)

In [ ]:
Hydro_df = pd.DataFrame()
for filename in os.listdir("files/HydroDF/"):
        df = pd.read_csv(os.path.join("files/HydroDF/", filename))
        Hydro_df = pd.concat([Hydro_df, df], ignore_index=True)

#print the number of rows and columns in the dataframe
print(f"Number of rows: {len(Hydro_df)}")
print(f"Number of columns: {len(Hydro_df.columns)}")


Hydro_df.head()

NameError: name 'pd' is not defined

In [ ]:
#save df
Hydro_df.to_csv("data/HydroDF/HydroDF.csv", index=False)

Hydro_df.head()

,site_no,flow_cfs,SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
2001-10-01,10133800,6.00,0.0,41455.59,0.0,364.48,0.0,25.83,6.72,706.04,...,272.175708,6.850663,0.247913,77517.134986,200.841528,0.003987,16.180751,0.0,0.319043,-0.057401
2001-10-02,10133800,6.00,0.0,41290.54,0.0,349.20,0.0,24.35,6.37,718.74,...,264.058493,5.492923,0.251043,77279.626107,200.452388,0.003330,15.442275,0.0,2.752252,-2.093241
2001-10-03,10133800,6.00,0.0,41125.82,0.0,368.63,0.0,22.82,3.20,566.91,...,243.925945,0.000000,0.244892,77251.491727,197.763217,0.002488,13.005251,0.0,3.347992,-0.219512
2001-10-04,10133800,6.03,0.0,40961.44,0.0,367.77,0.0,21.55,1.90,525.33,...,248.301808,0.000000,0.243444,77024.933881,196.978764,0.002719,12.222909,0.0,4.624836,0.185733
2001-10-05,10133800,6.87,0.0,40797.44,0.0,380.22,0.0,20.21,-0.85,429.58,...,233.020334,0.000000,0.208923,77062.676582,194.535973,0.002861,9.512823,0.0,-0.220614,-1.173468
